# 7.7 填充的位置會被當線索嗎？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：補齊長度像空椅子：不是新客人**

attention 別讀 PAD，loss 也別考 PAD。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/padding.svg")))

**先想一想：**只遮loss，不遮attention，左PAD會影響答案嗎？

左PAD出現在真實token之前，單純causal mask會允許看到它。需要有效key mask，並讓真實token的位置ID保持原來的0,1,2。

**把直覺寫成數學：**allowed=causal AND key_valid；position ID與物理padding位置不是同一件事。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

model = TinyLM(ModelConfig(vocab_size=10, width=8))
base = model(torch.tensor([[1, 2, 3]]))["logits"]
ids = torch.tensor([[0, 0, 1, 2, 3]])
valid = ids != 0
pos = torch.tensor([[0, 0, 0, 1, 2]])
actual = model(ids, valid=valid, positions=pos)["logits"][:, 2:]
print((base - actual).abs().max().item())
assert torch.allclose(base, actual, atol=1e-6)

**如何讀結果：**無有效key的PAD query需處理全-∞行避免NaN。基準已有此處理。

**只改一件事：**只取消valid mask，觀察等價斷言是否失敗。
